# Head Function Classifier

Automatically classify attention heads into functional categories:
previous-token, induction, copying, and positional heads.

## Why This Matters

Attention head function classifier reveals how heads route information between positions. Understanding attention mechanics is central to reverse-engineering transformer circuits, as attention patterns determine what information each component can access and transform.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.head_function_classifier import (
    classify_previous_token, classify_induction, classify_copying,
    classify_positional, head_function_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
# Use tokens with repeats for induction detection
tokens = jnp.array([1, 15, 30, 45, 15, 30, 45])
print('Model ready')

## Previous-Token Heads

Which heads primarily attend to the immediately preceding token?

In [ ]:
result = classify_previous_token(model, tokens, layer=0)
for h in result['per_head']:
    flag = ' ← PREV TOKEN' if h['is_previous_token'] else ''
    bar = '█' * int(h['previous_token_score'] * 30)
    print(f"  Head {h['head']}: score={h['previous_token_score']:.4f} {bar}{flag}")

## Induction Heads

Which heads implement the induction pattern (attend to tokens that
follow a previous occurrence of the current token)?

In [ ]:
result = classify_induction(model, tokens, layer=1)
for h in result['per_head']:
    flag = ' ← INDUCTION' if h['is_induction'] else ''
    bar = '█' * int(h['induction_score'] * 30)
    print(f"  Head {h['head']}: score={h['induction_score']:.4f} {bar}{flag}")

## Copying Heads

Which heads copy source tokens to the output (OV circuit analysis)?

In [ ]:
result = classify_copying(model, tokens, layer=0, top_k=5)
for h in result['per_head']:
    print(f"  Head {h['head']}: copy={h['copy_score']:.4f} "
          f"advantage={h['copy_advantage']:.4f}")
    for tok_id, score in h['top_copied_tokens']:
        print(f"    token {tok_id}: {score:.4f}")

## Positional Heads

Which heads have content-independent (positional) attention patterns?

In [ ]:
result = classify_positional(model, tokens, layer=0)
for h in result['per_head']:
    flag = ' ← POSITIONAL' if h['is_positional'] else ''
    bar = '█' * int(h['positional_score'] * 30)
    print(f"  Head {h['head']}: score={h['positional_score']:.4f} "
          f"var={h['mean_pattern_variance']:.6f} {bar}{flag}")

## Full Classification Summary

In [ ]:
result = head_function_summary(model, tokens)
for layer_info in result['per_layer']:
    print(f"\nLayer {layer_info['layer']}:")
    print(f"  prev_token: {layer_info['n_previous_token']}, "
          f"induction: {layer_info['n_induction']}, "
          f"positional: {layer_info['n_positional']}")
    for h in layer_info['per_head']:
        print(f"  Head {h['head']}: {', '.join(h['roles'])}")